# 07 — RAG Evaluation and Tracing

**First Finance - Arnaud Demes**  
**Day 1 · 16:00–16:45 · 15 minutes deck + 30 minutes guided notebook**

Lesson 06 produced a filtered hybrid retriever. This laboratory asks whether it
retrieved the right evidence, whether the answer used that evidence, and which stage
failed when the complete system did not meet the contract.

## Learning objectives

By the end of the laboratory, you can:

1. treat a golden set as versioned product data;
2. separate retrieval metrics from answer metrics;
3. trace the real eligibility → retrieval → generation path with local MLflow;
4. compare two configurations on the same cases;
5. classify a failure before proposing a fix; and
6. explain when optional Ragas judges add value and cost.

## Where this fits

```text
Lessons 04–05             Lesson 06                         Lesson 07
context + evidence  →  retrieve + fuse + rerank  →  evaluate + trace + compare
```

| Minutes | Work | Observable output |
|---:|---|---|
| 0–4 | Load the versioned golden set | case map |
| 4–9 | Run one baseline case | separated metric bars |
| 9–14 | Evaluate all cases | case-by-metric heatmap |
| 14–20 | Inspect one MLflow trace | real stage path and timings |
| 20–25 | Compare two configurations | aligned run comparison |
| 25–28 | Diagnose one failure | failure-stage table |
| 28–30 | Verify and hand off | PASS marker + knowledge check |

Ragas appears only after the main PASS marker. The lesson never hides its baseline
behind an LLM judge.

In [ ]:
from __future__ import annotations

import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch

from finai_academy.chunking import contextualize_chunks, structure_aware_chunks
from finai_academy.documents import load_source_manifest, parse_html, parse_pdf
from finai_academy.evaluation import evaluate_case, load_evaluation_cases
from finai_academy.hybrid_retrieval import (
    DenseIndex,
    DeterministicTeachingEmbeddings,
    IndexedPassage,
    KeywordIndex,
)
from finai_academy.lesson_support import compact_manifest_labels
from finai_academy.measurement import NullStageObserver
from finai_academy.mlflow_evaluation import (
    EvaluationConfiguration,
    EvaluationPrediction,
    run_mlflow_evaluation,
)
from finai_academy.providers import (
    check_provider_configuration,
    create_chat_model,
    create_embeddings,
)
from finai_academy.ragas_evaluation import (
    RagasEvaluationRow,
    RecordedRagasJudge,
    evaluate_with_ragas,
)
from finai_academy.retrieval_pipeline import retrieve_evidence
from finai_academy.settings import Settings

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "assets" / "course-data"
MLFLOW_ROOT = Path(
    os.getenv(
        "FINAI_MLFLOW_DIR",
        str(Path(tempfile.gettempdir()) / "finai-lesson07-mlflow"),
    )
)

COLORS = {
    "navy": "#051C2A", "blue": "#1F40CB", "cyan": "#00A2EB",
    "orange": "#F07D00", "green": "#2E8B57", "grey": "#64748B",
    "light": "#E8EEF5", "red": "#C43D3D",
}
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.titleweight": "bold"})

live_mode = os.getenv("FINAI_LIVE_MODE", "0") == "1"
if live_mode:
    settings = Settings.from_environment()
    problems = check_provider_configuration(settings)
    if problems:
        raise RuntimeError(" ".join(problems))
    embeddings = create_embeddings(settings)
    chat_model = create_chat_model(settings)
    provider = settings.provider
    chat_model_name = settings.chat_model
    embedding_model_name = settings.embedding_model
else:
    embeddings = DeterministicTeachingEmbeddings()
    chat_model = None
    provider = "offline"
    chat_model_name = "recorded-answer-v1"
    embedding_model_name = embeddings.model_name

print(f"Evaluation runtime: {provider} / {chat_model_name} / {embedding_model_name}")
print(f"Local MLflow store: {MLFLOW_ROOT}")

## 1. Versioned golden set

A golden set is not a list of demo prompts. Each case fixes the question, eligibility
filters, expected evidence IDs, expected facts, abstention requirement and diagnostic
tags. The JSON file and its SHA-256 hash are registered in the course manifest.

The set deliberately contains positive questions and unsupported questions. A system
that always returns a fluent answer cannot pass the latter.

In [ ]:
sources = load_source_manifest(DATA_ROOT / "manifest.json")
assert all(source.verify_fixture(REPO_ROOT) for source in sources)

chunks = []
for source in sources:
    fixture_path = REPO_ROOT / source.fixture_path
    blocks = (
        parse_html(fixture_path, source)
        if fixture_path.suffix == ".html"
        else parse_pdf(fixture_path, source)
    )
    chunks.extend(contextualize_chunks(structure_aware_chunks(blocks, max_chars=220)))

passages = tuple(
    IndexedPassage(
        passage_id=chunk.chunk_id,
        company=chunk.company,
        period=chunk.period,
        document_type=chunk.document_type,
        section=" > ".join(chunk.section_path) or "Document",
        text=chunk.text,
        source_url=chunk.source_url,
    )
    for chunk in chunks
)
known_evidence_ids = {passage.passage_id for passage in passages}
cases = load_evaluation_cases(
    DATA_ROOT / "evaluation" / "rag_cases_v1.json",
    known_evidence_ids=known_evidence_ids,
)
assert len(cases) == 8

keyword_index = KeywordIndex(passages)
dense_index = DenseIndex(
    passages,
    embeddings,
    provider=provider,
    model=embedding_model_name,
    chunking_strategy="contextual-structure-v1-max220",
)

case_frame = pd.DataFrame(
    {
        "case_id": case.case_id,
        "company": case.filters.company,
        "expected_evidence": len(case.expected_evidence_ids),
        "requires_abstention": case.requires_abstention,
        "tags": ", ".join(case.tags),
    }
    for case in cases
)
display(case_frame)

counts = case_frame.groupby(["company", "requires_abstention"], dropna=False).size()
fig, ax = plt.subplots(figsize=(11.5, 4.5))
labels = [f"{company}\n{'abstain' if abstain else 'answer'}" for company, abstain in counts.index]
bars = ax.bar(labels, counts.values, color=[COLORS["orange"] if value else COLORS["blue"] for _, value in counts.index])
ax.bar_label(bars, padding=4)
ax.set_ylabel("golden-set cases")
ax.set_title("Figure 1 — The golden set covers answer and abstention behavior", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print("Golden set loaded: rag-cases-v1 / 8 cases / 7 known evidence IDs")

## 2. Retrieval metrics and answer metrics

Retrieval recall@k and reciprocal rank ask whether expected evidence appeared and how
early. Filter correctness asks whether the returned passages respected eligibility.
Citation correctness and grounded-fact coverage inspect the answer. Abstention
correctness tests whether the application declined unsupported work.

One good retrieval can still feed a bad answer. One well-written answer can still hide a
retrieval miss. Keep these axes separate.

In [ ]:
OFFLINE_ANSWERS = {
    "nvda-direct-fact": "Data Center generated $193.7 billion [NVDA-2026-10K-EXCERPT-CONTEXTUAL-003].",
    "nvda-exact-number": "Data Center reported $193.7 billion [NVDA-2026-10K-EXCERPT-CONTEXTUAL-003].",
    "nvda-semantic-paraphrase": "Data Center represented most of the reported year-on-year expansion [NVDA-2026-10K-EXCERPT-CONTEXTUAL-004].",
    "nvda-filter-safety": "NVIDIA fiscal 2026 revenue was $215.9 billion, up 65% [NVDA-2026-10K-EXCERPT-CONTEXTUAL-001].",
    "schneider-filter-safety": "Schneider Electric FY2025 revenue was EUR 40.2bn [SU-2025-FY-EXCERPT-CONTEXTUAL-001].",
    "cross-company-leakage": "Insufficient evidence for NVIDIA Energy Management growth.",
    "multi-evidence-comparison": "NVIDIA total revenue was $215.9 billion [NVDA-2026-10K-EXCERPT-CONTEXTUAL-001], versus $193.7 billion for Data Center [NVDA-2026-10K-EXCERPT-CONTEXTUAL-003].",
    "insufficient-evidence": "The provided evidence does not establish fair value.",
}
ANSWER_CACHE = {}

baseline_configuration = EvaluationConfiguration(
    configuration_id="hybrid-rrf-1-1",
    dataset_version="rag-cases-v1",
    provider=provider,
    chat_model=chat_model_name,
    embedding_model=embedding_model_name,
    index_version=dense_index.version.corpus_hash,
    prompt_version="answer-with-citations-v1",
    candidate_k=4,
    final_k=2,
    rrf_weights={"keyword": 1.0, "dense": 1.0},
)

def generated_answer(case, retrieval, configuration):
    evidence_key = tuple(hit.passage.passage_id for hit in retrieval.reranked_hits)
    cache_key = (case.case_id, evidence_key, configuration.prompt_version)
    if cache_key in ANSWER_CACHE:
        return ANSWER_CACHE[cache_key]
    if not live_mode:
        answer = OFFLINE_ANSWERS[case.case_id]
    else:
        evidence = "\n\n".join(
            f"[{hit.passage.passage_id}] {hit.passage.text}"
            for hit in retrieval.reranked_hits
        )
        response = chat_model.invoke(
            [
                (
                    "system",
                    "Answer only from supplied evidence. Cite stable passage IDs in square brackets. If evidence is insufficient, say so explicitly.",
                ),
                ("human", f"Question: {case.question}\n\nEvidence:\n{evidence}"),
            ]
        )
        content = response.content
        answer = content.strip() if isinstance(content, str) else str(content)
    ANSWER_CACHE[cache_key] = answer
    return answer

def make_predictor(configuration, store):
    def predict(case, observer):
        retrieval = retrieve_evidence(
            case.question,
            keyword_index=keyword_index,
            dense_index=dense_index,
            filters=case.filters,
            candidate_k=configuration.candidate_k,
            final_k=configuration.final_k,
            weights=configuration.rrf_weights,
            observer=observer,
        )
        with observer.span("context", inputs={"final_k": configuration.final_k}):
            contexts = tuple(hit.passage.text for hit in retrieval.reranked_hits)
        with observer.span("generation", inputs={"prompt_version": configuration.prompt_version}):
            answer = generated_answer(case, retrieval, configuration)
        prediction = EvaluationPrediction(retrieval=retrieval, answer=answer, contexts=contexts)
        store[case.case_id] = prediction
        return prediction
    return predict

example_case = cases[0]
example_store = {}
example_prediction = make_predictor(baseline_configuration, example_store)(
    example_case,
    NullStageObserver(),
)
example_result = evaluate_case(example_case, example_prediction.retrieval, example_prediction.answer)

example_metrics = {
    "retrieval recall@k": example_result.retrieval_recall_at_k,
    "reciprocal rank": example_result.reciprocal_rank,
    "filter correctness": example_result.filter_correctness,
    "citation correctness": example_result.citation_correctness,
    "grounded facts": example_result.grounded_fact_coverage,
    "abstention": example_result.abstention_correctness,
}
fig, ax = plt.subplots(figsize=(11.5, 4.8))
bars = ax.bar(list(example_metrics), list(example_metrics.values()), color=[COLORS["blue"]] * 3 + [COLORS["green"]] * 3)
ax.bar_label(bars, fmt="%.2f", padding=4)
ax.set_ylim(0, 1.12)
ax.tick_params(axis="x", rotation=18)
ax.set_title("Figure 2 — One case, two quality axes", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print("Baseline case evaluated:", example_case.case_id, "→", example_result.failure_stage)

## 3. Full deterministic metric suite

The full run uses the real Lesson 06 pipeline. The offline path has fixed answers so the
teaching baseline is exact. Ollama and OpenAI generate live answers; their metric values
are observations, while trace shape, metadata, finite scores and case alignment remain
hard contracts.

In [ ]:
baseline_predictions = {}
baseline_summary = run_mlflow_evaluation(
    tracking_path=MLFLOW_ROOT,
    experiment_name="finai-lesson07-rag-evaluation",
    configuration=baseline_configuration,
    cases=cases,
    predict_fn=make_predictor(baseline_configuration, baseline_predictions),
)

def case_metrics_frame(configuration, predictions):
    rows = []
    for case in cases:
        prediction = predictions[case.case_id]
        result = evaluate_case(case, prediction.retrieval, prediction.answer)
        rows.append(
            {
                "case_id": case.case_id,
                "configuration_id": configuration.configuration_id,
                "retrieval_recall_at_k": result.retrieval_recall_at_k,
                "reciprocal_rank": result.reciprocal_rank,
                "filter_correctness": result.filter_correctness,
                "citation_correctness": result.citation_correctness,
                "grounded_fact_coverage": result.grounded_fact_coverage,
                "abstention_correctness": result.abstention_correctness,
                "failure_stage": result.failure_stage,
            }
        )
    return pd.DataFrame(rows)

baseline_frame = case_metrics_frame(baseline_configuration, baseline_predictions)
display(baseline_frame)

metric_columns = [
    "retrieval_recall_at_k", "reciprocal_rank", "filter_correctness",
    "citation_correctness", "grounded_fact_coverage", "abstention_correctness",
]
matrix = baseline_frame.set_index("case_id")[metric_columns]
fig, ax = plt.subplots(figsize=(13.5, 6.3))
image = ax.imshow(matrix.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(metric_columns)), [name.replace("_", "\n") for name in metric_columns])
ax.set_yticks(range(len(matrix.index)), matrix.index)
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        value = matrix.iloc[row, column]
        ax.text(column, row, f"{value:.2f}", ha="center", va="center", color="white" if value > 0.6 else COLORS["navy"], weight="bold")
ax.set_title("Figure 3 — Case-level metrics reveal different failure patterns", loc="left")
fig.colorbar(image, ax=ax, fraction=0.025, pad=0.03, label="metric value")
plt.tight_layout()
plt.show()

## 4. MLflow trace

The local store is SQLite plus a local artifact directory—no Docker and no hosted
service. One MLflow run represents one configuration. One trace represents one golden-set
case. The root trace contains the question, filters, retrieved IDs, scores and answer;
child spans expose eligibility, keyword, dense, fusion, rerank, context and generation.

In [ ]:
baseline_run = mlflow.get_run(baseline_summary.run_id)
baseline_experiment_id = baseline_run.info.experiment_id
baseline_traces = mlflow.search_traces(
    run_id=baseline_summary.run_id,
    locations=[baseline_experiment_id],
    return_type="list",
    flush=True,
)
assert len(baseline_traces) == len(cases)
span_names = {span.name for trace in baseline_traces for span in trace.data.spans}
required_span_names = {"eligibility", "keyword", "dense", "fusion", "rerank", "context", "generation"}
assert required_span_names <= span_names

trace_case = cases[0]
trace_prediction = baseline_predictions[trace_case.case_id]
trace_stages = ["eligibility", "keyword", "dense", "fusion", "rerank", "context", "generation"]
stage_durations = {
    name: trace_prediction.retrieval.stage_measurements[name].duration_ms
    for name in ("eligibility", "keyword", "dense", "fusion", "rerank")
}
stage_durations.update({"context": 0.0, "generation": 0.0})

fig, ax = plt.subplots(figsize=(15, 4.3))
ax.set_xlim(0, 15)
ax.set_ylim(0, 3)
ax.axis("off")
for index, stage in enumerate(trace_stages):
    x = 0.25 + index * 2.1
    if index:
        ax.annotate("", xy=(x - 0.08, 1.35), xytext=(x - 0.36, 1.35), arrowprops={"arrowstyle": "->", "lw": 2, "color": COLORS["cyan"]})
    box = FancyBboxPatch((x, 0.78), 1.75, 1.15, boxstyle="round,pad=0.04", facecolor=COLORS["navy"] if stage in {"dense", "generation"} else "white", edgecolor=COLORS["blue"], linewidth=2)
    ax.add_patch(box)
    text_color = "white" if stage in {"dense", "generation"} else COLORS["navy"]
    ax.text(x + 0.875, 1.48, stage, ha="center", va="center", weight="bold", color=text_color)
    duration = stage_durations[stage]
    ax.text(x + 0.875, 1.12, f"{duration:.2f} ms" if stage not in {"context", "generation"} else "traced", ha="center", va="center", fontsize=8.5, color=text_color)
ax.set_title("Figure 4 — One case trace connects real application stages", loc="left")
plt.tight_layout()
plt.show()
print(f"MLflow traces recorded: {len(baseline_traces)}")
print("MLflow run ID:", baseline_summary.run_id)

## 5. Compare two configurations

Change one variable and keep the dataset, prompt, model, index and budgets visible. Here
only the keyword RRF weight changes from 1 to 3. The run comparison is useful even when
aggregate scores tie: per-case ranks and failure rows remain aligned.

In [ ]:
weighted_configuration = EvaluationConfiguration(
    configuration_id="hybrid-rrf-3-1",
    dataset_version=baseline_configuration.dataset_version,
    provider=provider,
    chat_model=chat_model_name,
    embedding_model=embedding_model_name,
    index_version=dense_index.version.corpus_hash,
    prompt_version=baseline_configuration.prompt_version,
    candidate_k=4,
    final_k=2,
    rrf_weights={"keyword": 3.0, "dense": 1.0},
)
weighted_predictions = {}
weighted_summary = run_mlflow_evaluation(
    tracking_path=MLFLOW_ROOT,
    experiment_name="finai-lesson07-rag-evaluation",
    configuration=weighted_configuration,
    cases=cases,
    predict_fn=make_predictor(weighted_configuration, weighted_predictions),
)
weighted_frame = case_metrics_frame(weighted_configuration, weighted_predictions)

comparison = pd.DataFrame(
    [
        {"configuration": baseline_configuration.configuration_id, **baseline_summary.metrics},
        {"configuration": weighted_configuration.configuration_id, **weighted_summary.metrics},
    ]
).set_index("configuration")
display(comparison)

selected_metrics = ["retrieval_recall_at_k", "citation_correctness", "grounded_fact_coverage", "abstention_correctness"]
x = np.arange(len(selected_metrics))
fig, ax = plt.subplots(figsize=(12.5, 5.2))
width = 0.36
ax.bar(x - width / 2, comparison.loc[baseline_configuration.configuration_id, selected_metrics], width, label=baseline_configuration.configuration_id, color=COLORS["blue"])
ax.bar(x + width / 2, comparison.loc[weighted_configuration.configuration_id, selected_metrics], width, label=weighted_configuration.configuration_id, color=COLORS["orange"])
ax.set_xticks(x, [name.replace("_", "\n") for name in selected_metrics])
ax.set_ylim(0, 1.08)
ax.set_ylabel("aggregate metric")
ax.set_title("Figure 5 — Compare configurations on the same cases", loc="left")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

rank_case_id = "multi-evidence-comparison"
rank_rows = []
rank_passages = {
    hit.passage.passage_id: hit.passage
    for predictions in (baseline_predictions, weighted_predictions)
    for hit in predictions[rank_case_id].retrieval.reranked_hits
}
rank_labels = compact_manifest_labels(list(rank_passages.values()))
for configuration, predictions in ((baseline_configuration, baseline_predictions), (weighted_configuration, weighted_predictions)):
    ids = [hit.passage.passage_id for hit in predictions[rank_case_id].retrieval.reranked_hits]
    for rank, passage_id in enumerate(ids, start=1):
        rank_rows.append({"configuration": configuration.configuration_id, "passage_id": rank_labels[passage_id], "rank": rank})
rank_frame = pd.DataFrame(rank_rows)
fig, ax = plt.subplots(figsize=(10.5, 4.5))
for index, configuration_id in enumerate(rank_frame.configuration.unique()):
    rows = rank_frame[rank_frame.configuration == configuration_id]
    ax.scatter([index] * len(rows), rows["rank"], s=130, color=COLORS["blue"] if index == 0 else COLORS["orange"])
    for _, row in rows.iterrows():
        ax.text(index + 0.06, row["rank"], row["passage_id"], va="center")
ax.set_xticks(range(2), rank_frame.configuration.unique())
ax.set_yticks([1, 2])
ax.invert_yaxis()
ax.set_ylabel("final evidence rank")
ax.set_title("Figure 6 — Aggregate ties can hide rank-level changes", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Failure lab

## 6. Failure analysis

The negative cases expose a real design gap: metadata eligibility can find NVIDIA
passages even when those passages do not answer a valuation or Energy Management
question. The current pipeline therefore returns evidence instead of abstaining.

Do not “fix” this by lowering a similarity threshold until the example looks right.
Classify the failure as **abstention / evidence sufficiency**, then add and evaluate an
explicit sufficiency gate in the capstone.

In [ ]:
failure_rows = pd.DataFrame(
    [dict(row) for row in baseline_summary.failure_rows]
    + [dict(row) for row in weighted_summary.failure_rows]
)
display(failure_rows)

failure_counts = failure_rows.groupby(["configuration_id", "failure_stage"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(11.5, 4.8))
failure_counts.plot(kind="bar", ax=ax, color=[COLORS["red"], COLORS["orange"], COLORS["blue"]][:len(failure_counts.columns)])
ax.set_ylabel("failed cases")
ax.set_xlabel("")
ax.set_title("Figure 7 — Failure stages direct the next engineering change", loc="left")
ax.legend(title="failure stage", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("Failure classified:", failure_rows.iloc[0]["failure_stage"])

## Verification

The PASS contract checks the dataset, two aligned configurations, persisted MLflow
traces, required span names, complete reproducibility metadata, finite metrics and an
explicit failure classification. Offline additionally checks the six positive cases.
Live provider metric values remain observations.

In [ ]:
positive_ids = {case.case_id for case in cases if not case.requires_abstention}
offline_positive_recall = baseline_frame[baseline_frame.case_id.isin(positive_ids)].retrieval_recall_at_k.eq(1.0).all()
required_parameters = {"dataset_version", "provider", "chat_model", "embedding_model", "index_version", "prompt_version", "candidate_k", "final_k", "rrf_weights"}
checks = {
    "8 versioned cases": len(cases) == 8,
    "two aligned configurations": set(baseline_frame.case_id) == set(weighted_frame.case_id),
    "8 baseline traces": len(baseline_traces) == len(cases),
    "required spans": required_span_names <= span_names,
    "reproducibility metadata": required_parameters <= set(baseline_summary.parameters),
    "finite aggregate metrics": np.isfinite(list(baseline_summary.metrics.values())).all(),
    "failure stage classified": bool(baseline_summary.failure_rows) and all(row["failure_stage"] != "none" for row in baseline_summary.failure_rows),
    "offline positive retrieval": (not live_mode and offline_positive_recall) or live_mode,
}

fig, ax = plt.subplots(figsize=(12.5, 5.2))
ax.axis("off")
for row, (label, passed) in enumerate(checks.items()):
    y = len(checks) - row
    ax.text(0.05, y, "PASS" if passed else "FAIL", color=COLORS["green"] if passed else COLORS["red"], weight="bold", va="center")
    ax.text(0.20, y, label, color=COLORS["navy"], va="center")
ax.set_xlim(0, 1)
ax.set_ylim(0, len(checks) + 1)
ax.set_title("Figure 8 — Evaluation and tracing acceptance contract", loc="left")
plt.tight_layout()
plt.show()

assert all(checks.values()), checks
print("PASS — RAG evaluation and tracing verified")

## Knowledge check

1. Why can retrieval recall@k be 1.0 while citation correctness is 0.0?  
   **Answer:** the correct passage was retrieved, but the answer omitted it or cited a
   different ID.

2. Why should the two configuration runs share the same golden set and prompt version?  
   **Answer:** otherwise the comparison changes more than one variable and cannot support
   a causal engineering decision.

3. What does a trace add beyond aggregate metrics?  
   **Answer:** it connects one input to stage inputs, outputs, timing and the final failure.

4. Why is Ragas after the deterministic baseline?  
   **Answer:** judge metrics add model cost and variance; first-principles metrics remain
   the stable debugging contract.

## Challenge

Add an evidence-sufficiency gate between reranking and generation. It must abstain on the
two negative cases without reducing recall on the six positive cases. Log the gate as a
new span and compare a third MLflow configuration. Do not change the golden set to make
the gate pass.

## Capstone integration

The Financial Analyst Copilot now owns:

- a versioned NVIDIA/Schneider golden set;
- deterministic retrieval and answer metrics;
- a local MLflow experiment with one trace per case;
- configuration comparison; and
- a classified abstention gap for Day 2.

Lesson 08 will convert the measured pipeline into a stateful LangGraph workflow rather
than hiding the stages inside an unconstrained agent.

## Optional Ragas comparison

Ragas can add model-judged **context recall** and **faithfulness**. The judge provider and
model must be supplied explicitly; the adapter never initializes a default model. This
section is optional because its output depends on the chosen judge and consumes model
tokens.

In [ ]:
ragas_rows = tuple(
    RagasEvaluationRow(
        case_id=case.case_id,
        user_input=case.question,
        retrieved_contexts=baseline_predictions[case.case_id].contexts,
        response=baseline_predictions[case.case_id].answer,
        reference_answer=(
            "; ".join(case.expected_facts)
            if case.expected_facts
            else "The system should abstain because the evidence is insufficient."
        ),
    )
    for case in cases
)
ragas_result = evaluate_with_ragas(
    ragas_rows,
    judge=RecordedRagasJudge(metrics={}),
)

fig, ax = plt.subplots(figsize=(11.5, 4.2))
ax.axis("off")
steps = [
    (0.08, "Deterministic baseline", "retrieval · citations · facts"),
    (0.39, "Explicit Ragas judge", "context recall · faithfulness"),
    (0.70, "Decision", "gain vs cost + variance"),
]
for x, title, subtitle in steps:
    box = FancyBboxPatch((x, 0.35), 0.23, 0.38, boxstyle="round,pad=0.02", facecolor="white", edgecolor=COLORS["blue"], linewidth=2)
    ax.add_patch(box)
    ax.text(x + 0.115, 0.58, title, ha="center", weight="bold", color=COLORS["navy"])
    ax.text(x + 0.115, 0.45, subtitle, ha="center", fontsize=8.5, color=COLORS["grey"])
for x in (0.33, 0.64):
    ax.annotate("", xy=(x + 0.045, 0.54), xytext=(x, 0.54), arrowprops={"arrowstyle": "->", "lw": 2, "color": COLORS["orange"]})
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Figure 9 — Ragas is an optional comparison, not the baseline", loc="left")
plt.tight_layout()
plt.show()
print("Optional Ragas status:", ragas_result.status, "/ judge:", ragas_result.judge_provider, ragas_result.judge_model)

## Recap

- Version the golden set with the corpus, index and prompt.
- Keep retrieval quality, answer quality and abstention as separate metrics.
- Use one MLflow run per configuration and one trace per case.
- Diagnose the failure stage before changing the application.
- Add Ragas only with an explicit judge and after the reproducible baseline.